# Experiment 1 v2 — Notebook 02: DAE Search

**Dataset:** `CICIDS2017`  
**Scientific scope:** within-dataset binary detection of known/documented attacks using the frozen Experiment 1 v2 representation.  
**Upstream authority:** PRE-00 → PRE-01 → PRE-02. This notebook consumes frozen artifacts and never reconstructs preprocessing, partitions, feature policy, taxonomy, or corruption groups.

This notebook is designed for Google Colab and is independently executable **after all prerequisite stages have passed**.


## Scientific stage boundary

Weights may be updated only from `benign_weight_train`. `benign_early_stop` may influence early stopping / best-checkpoint selection only. `benign_threshold_calibration` may influence threshold estimation only. Development observations may influence configuration selection or diagnostics but never weight/preprocessing/threshold fitting.
**Final-test outcomes remain sealed in this notebook.**

Attack-family labels are diagnostic metadata only. They are never network inputs, loss targets, threshold inputs, or model-selection criteria.


In [1]:
# ============================================================
# 1. EDITABLE CONFIGURATION
# ============================================================
from pathlib import Path

DATASET_NAME = 'CICIDS2017'
DATASET_FOLDER = 'CICIDS2017'
DATASET_PREFIX = 'CICIDS2017'
PROTOCOL_VERSION = "Experiment_1_v2"
NOTEBOOK_STAGE_NUMBER = 2
NOTEBOOK_STAGE = "Notebook_02_dae_search"
NOTEBOOK_VERSION = '1.0.0'
CODE_VERSION = 'experiment1_v2_model_core_1.0.0'
PROJECT_FOLDER_NAME = "EXPERIMENT1_V2"

# Optional explicit overrides. Leave None for conservative discovery.
PROJECT_ROOT_OVERRIDE = None
PRE02_ROOT_OVERRIDE = None
GLOBAL_DIR_OVERRIDE = None
STAGE_ROOT_OVERRIDES = {}  # e.g. {"notebook03_confirmation": "/content/..."}

PRIMARY_DRIVE_MOUNTPOINT = Path("/content/drive")
FALLBACK_DRIVE_MOUNTPOINT_BASE = Path("/content/exp1v2_google_drive")

# Formal is the authoritative experiment. Smoke outputs are kept separate.
RUN_MODE = "formal"  # allowed: "formal", "smoke"
SMOKE_UNIQUE_COMPLETED = 2
MAX_SEARCH_ATTEMPTS = 250

print("Dataset:", DATASET_NAME)
print("Stage:", NOTEBOOK_STAGE)
print("Notebook version:", NOTEBOOK_VERSION)


Dataset: CICIDS2017
Stage: Notebook_02_dae_search
Notebook version: 1.0.0


In [2]:
# ============================================================
# 2. RUNTIME DEPENDENCIES, IMPORTS, ENVIRONMENT
# ============================================================
import sys, subprocess, importlib.util

_REQUIRED = {
    "optuna": "optuna>=3.6,<5",
    "pyarrow": "pyarrow>=14",
    "sklearn": "scikit-learn>=1.3",
    "scipy": "scipy>=1.10",
    "matplotlib": "matplotlib>=3.7",
}
for module_name, pip_spec in _REQUIRED.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required. Use a standard Google Colab runtime with PyTorch installed.")

import os, gc, math, json, time, random, hashlib, platform, shutil, traceback, inspect
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import sklearn
from sklearn.metrics import (
    matthews_corrcoef, recall_score, f1_score, precision_score,
    balanced_accuracy_score, precision_recall_curve, auc,
    average_precision_score, roc_auc_score, roc_curve,
)
import scipy
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import joblib
import optuna
from optuna.trial import TrialState
import torch
from torch import nn

EXECUTION_UTC = datetime.now(timezone.utc).isoformat()
ENVIRONMENT_INFO = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pa.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "matplotlib": __import__("matplotlib").__version__,
    "optuna": optuna.__version__,
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "execution_utc": EXECUTION_UTC,
}
display(pd.DataFrame([ENVIRONMENT_INFO]).T.rename(columns={0: "value"}).head(30))


,value
python,"3.13.15 (main, Aug 6 2026, 11:06:22) [GCC 13...."
platform,Linux-6.6.122+-x86_64-with-glibc2.39
numpy,2.1.3
pandas,2.2.3
pyarrow,23.0.1
scikit_learn,1.6.1
scipy,1.16.3
matplotlib,3.10.0
optuna,4.9.0
torch,2.11.0+cu128


In [3]:
# ============================================================
# 3. VERIFIED GOOGLE DRIVE / PROJECT BINDING
# ============================================================
def _find_mydrive(mountpoint):
    mountpoint = Path(mountpoint)
    for name in ["MyDrive", "My Drive"]:
        p = mountpoint / name
        if p.exists() and p.is_dir():
            return p
    return None


def _find_project(mydrive, project_name):
    if mydrive is None:
        return None
    direct = Path(mydrive) / project_name
    if direct.exists() and direct.is_dir():
        return direct.resolve()
    matches = []
    for dirpath, dirnames, _ in os.walk(str(mydrive), topdown=True, followlinks=True):
        dirnames[:] = [d for d in dirnames if d not in {".Trash", ".trash", "__pycache__", ".ipynb_checkpoints"}]
        if project_name in dirnames:
            matches.append((Path(dirpath) / project_name).resolve())
    matches = list(dict.fromkeys(map(str, matches)))
    if len(matches) == 1:
        return Path(matches[0])
    if len(matches) > 1:
        raise RuntimeError(f"Multiple {project_name!r} project roots found: {matches}. Set PROJECT_ROOT_OVERRIDE explicitly.")
    return None


def _fresh_mountpoint(base):
    base = Path(base)
    for i in range(20):
        candidate = base if i == 0 else Path(f"{base}_{i}")
        if not candidate.exists() or (candidate.is_dir() and not any(candidate.iterdir())):
            return candidate
    raise RuntimeError("No fresh alternate Google Drive mountpoint is available under /content.")


try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {PROJECT_ROOT}")
    DRIVE_SEARCH_ROOT = PROJECT_ROOT.parent.parent
else:
    primary_mydrive = _find_mydrive(PRIMARY_DRIVE_MOUNTPOINT)
    primary_project = _find_project(primary_mydrive, PROJECT_FOLDER_NAME)
    if primary_project is not None:
        PROJECT_ROOT = primary_project
        DRIVE_SEARCH_ROOT = PRIMARY_DRIVE_MOUNTPOINT.resolve()
    else:
        if not IN_COLAB:
            raise RuntimeError("Verified Google Drive project is unavailable. Set PROJECT_ROOT_OVERRIDE explicitly.")
        alt = _fresh_mountpoint(FALLBACK_DRIVE_MOUNTPOINT_BASE)
        print("Mounting Google Drive at verified alternate mountpoint:", alt)
        drive.mount(str(alt), force_remount=False)
        alt_mydrive = _find_mydrive(alt)
        PROJECT_ROOT = _find_project(alt_mydrive, PROJECT_FOLDER_NAME)
        if PROJECT_ROOT is None:
            raise FileNotFoundError(f"Mounted Drive does not contain {PROJECT_FOLDER_NAME!r}. Set PROJECT_ROOT_OVERRIDE if renamed.")
        DRIVE_SEARCH_ROOT = alt.resolve()

DATASET_ROOT = PROJECT_ROOT / "DATASETS" / DATASET_FOLDER
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset root is missing: {DATASET_ROOT}")

GLOBAL_DIR = Path(GLOBAL_DIR_OVERRIDE).resolve() if GLOBAL_DIR_OVERRIDE else PROJECT_ROOT / "global"
MODELLING_ROOT = DATASET_ROOT / "modelling"
MODELLING_ROOT.mkdir(parents=True, exist_ok=True)
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("GLOBAL_DIR:", GLOBAL_DIR)
print("MODELLING_ROOT:", MODELLING_ROOT)


Mounting Google Drive at verified alternate mountpoint: /content/exp1v2_google_drive
Mounted at /content/exp1v2_google_drive
PROJECT_ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2
DATASET_ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017
GLOBAL_DIR: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/global
MODELLING_ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/modelling


In [4]:
# ============================================================
# 4. HASHING, IMMUTABLE JSON, CONTRACT, AND STAGE UTILITIES
# ============================================================
ALLOWED_PASS_STATUSES = {"PASS", "PASS_WITH_NONBLOCKING_WARNINGS"}


def hard_fail(message):
    print("EXPERIMENT 1 v2 MODEL-STAGE INTEGRITY: FAIL")
    print("Next authorized action: NONE until the blocking issue is corrected/versioned.")
    raise RuntimeError(str(message))


def require(condition, message):
    if not bool(condition):
        hard_fail(message)


def canonical_json_bytes(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode("utf-8")


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, block_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)
        f.write("\n")
    os.replace(tmp, path)


def atomic_write_csv(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_write_parquet(path, df):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False, compression="snappy")
    os.replace(tmp, path)


def substantive_hash(obj, excluded_keys):
    body = {k: v for k, v in obj.items() if k not in set(excluded_keys)}
    return sha256_bytes(canonical_json_bytes(body)), body


def verify_self_hashed_json(path, hash_field, extra_excluded=()):
    obj = read_json(path)
    excluded = {hash_field, "created_at_utc", *extra_excluded}
    computed, _ = substantive_hash(obj, excluded)
    stored = str(obj.get(hash_field, "")).lower()
    require(stored and computed.lower() == stored, f"Self-hash mismatch for {path}: computed={computed}, stored={stored}")
    return obj, computed


def freeze_contract(path, substantive, hash_field="contract_hash"):
    path = Path(path)
    expected_hash = sha256_bytes(canonical_json_bytes(substantive))
    if path.exists():
        obj = read_json(path)
        stored = str(obj.get(hash_field, "")).lower()
        body = {k: v for k, v in obj.items() if k not in {hash_field, "created_at_utc"}}
        require(stored == expected_hash.lower(), f"Existing contract hash mismatch: {path}")
        require(body == substantive, f"Existing contract substantive content differs: {path}")
        return obj, expected_hash
    obj = {**substantive, "created_at_utc": datetime.now(timezone.utc).isoformat(), hash_field: expected_hash}
    atomic_write_json(path, obj)
    return obj, expected_hash


def verify_contract(path, hash_field="contract_hash"):
    path = Path(path)
    require(path.exists(), f"Required global contract is missing: {path}. Run Notebook 01 first for this Experiment 1 v2 project.")
    return verify_self_hashed_json(path, hash_field)


def make_stage_manifest(substantive):
    h = sha256_bytes(canonical_json_bytes(substantive))
    return {**substantive, "created_at_utc": datetime.now(timezone.utc).isoformat(), "stage_manifest_hash": h}


def write_stage_manifest(path, substantive):
    obj = make_stage_manifest(substantive)
    path = Path(path)
    if path.exists():
        old, _ = verify_self_hashed_json(path, "stage_manifest_hash")
        old_body = {k: v for k, v in old.items() if k not in {"created_at_utc", "stage_manifest_hash"}}
        require(old_body == substantive, f"Existing stage manifest conflicts with current execution contract: {path}")
        return old
    atomic_write_json(path, obj)
    return obj


def load_stage_manifest(path):
    obj, _ = verify_self_hashed_json(path, "stage_manifest_hash")
    require(obj.get("final_status") in ALLOWED_PASS_STATUSES, f"Upstream stage is not PASS: {path}")
    require(obj.get("dataset") == DATASET_NAME, f"Stage manifest dataset mismatch: {path}")
    return obj


def dataset_matches(value):
    a = str(value).strip().casefold().replace("_", "-")
    b = DATASET_NAME.casefold().replace("_", "-")
    aliases = {b}
    if DATASET_NAME == "CICIDS2017": aliases |= {"cicids-2017", "cicids2017"}
    else: aliases |= {"unsw-nb15", "unswnb15"}
    return a in aliases


def discover_one(paths, description):
    unique = []
    seen = set()
    for p in paths:
        p = Path(p).resolve()
        if str(p) not in seen:
            unique.append(p); seen.add(str(p))
    require(len(unique) == 1, f"Expected exactly one {description}; found {len(unique)}: {[str(x) for x in unique]}")
    return unique[0]


def resolve_dependency_stage(stage_slug, require_formal=False):
    override = STAGE_ROOT_OVERRIDES.get(stage_slug)
    if override:
        root = Path(override).resolve()
        manifest_path = root / "stage_manifest.json"
        manifest = load_stage_manifest(manifest_path)
        for rel, expected in manifest.get("key_artifact_hashes", {}).items():
            ap = root / rel
            require(ap.exists(), f"Frozen upstream stage artifact is missing: {ap}")
            require(sha256_file(ap) == expected, f"Frozen upstream stage artifact hash mismatch: {ap}")
        return root, manifest
    base = MODELLING_ROOT / stage_slug
    candidates = []
    if base.exists():
        for p in base.rglob("stage_manifest.json"):
            try:
                obj, _ = verify_self_hashed_json(p, "stage_manifest_hash")
                if obj.get("dataset") != DATASET_NAME or obj.get("final_status") not in ALLOWED_PASS_STATUSES:
                    continue
                if require_formal and obj.get("run_mode") != "formal":
                    continue
                candidates.append(p.parent.resolve())
            except Exception:
                continue
    root = discover_one(candidates, f"valid upstream stage root for {stage_slug}")
    manifest = load_stage_manifest(root / "stage_manifest.json")
    for rel, expected in manifest.get("key_artifact_hashes", {}).items():
        ap = root / rel
        require(ap.exists(), f"Frozen upstream stage artifact is missing: {ap}")
        require(sha256_file(ap) == expected, f"Frozen upstream stage artifact hash mismatch: {ap}")
    return root, manifest


def create_artifact_inventory(root, output_path):
    root = Path(root); output_path = Path(output_path)
    rows = []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.resolve() == output_path.resolve() or p.name.endswith(".tmp"):
            continue
        rows.append({
            "relative_path": str(p.relative_to(root)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })
    atomic_write_csv(output_path, pd.DataFrame(rows))
    return rows


In [5]:
# ============================================================
# 5. DISCOVER / VERIFY PRE-02 AND FREEZE / VERIFY GLOBAL CONTRACTS
# ============================================================
ALLOW_GLOBAL_CONTRACT_CREATION = False


def discover_pre02_root():
    if PRE02_ROOT_OVERRIDE:
        root = Path(PRE02_ROOT_OVERRIDE).resolve()
        report_path = root / "reports" / "preprocessing_integrity_report.json"
        require(report_path.exists(), f"Explicit PRE02 root lacks integrity report: {root}")
        return root
    candidates = []
    for report_path in DATASET_ROOT.rglob("preprocessing_integrity_report.json"):
        try:
            report = read_json(report_path)
            if not dataset_matches(report.get("dataset")):
                continue
            if report.get("final_status") not in ALLOWED_PASS_STATUSES:
                continue
            root = report_path.parent.parent
            needed = [
                root / "manifests" / "preprocessing_policy_manifest.json",
                root / "manifests" / "transformed_representation_manifest.json",
                root / "manifests" / "corruption_group_manifest.json",
                root / "fitted_preprocessor.joblib",
            ]
            if all(p.exists() for p in needed):
                candidates.append(root.resolve())
        except Exception:
            continue
    return discover_one(candidates, "valid PRE-02 output root")


PRE02_ROOT = discover_pre02_root()
PRE02_INTEGRITY_PATH = PRE02_ROOT / "reports" / "preprocessing_integrity_report.json"
PRE02_INTEGRITY = read_json(PRE02_INTEGRITY_PATH)
require(PRE02_INTEGRITY.get("final_status") in ALLOWED_PASS_STATUSES, "PRE-02 integrity is not PASS/PASS_WITH_NONBLOCKING_WARNINGS.")
require(dataset_matches(PRE02_INTEGRITY.get("dataset")), "PRE-02 dataset mismatch.")

PREPROCESSING_POLICY_PATH = PRE02_ROOT / "manifests" / "preprocessing_policy_manifest.json"
REPRESENTATION_PATH = PRE02_ROOT / "manifests" / "transformed_representation_manifest.json"
CORRUPTION_PATH = PRE02_ROOT / "manifests" / "corruption_group_manifest.json"
FITTED_PREPROCESSOR_PATH = PRE02_ROOT / "fitted_preprocessor.joblib"

PREPROCESSING_POLICY = read_json(PREPROCESSING_POLICY_PATH)
# PRE-02 policy hash excludes operational_storage and timestamp.
policy_body = {k: v for k, v in PREPROCESSING_POLICY.items() if k not in {"preprocessing_policy_hash", "created_at_utc", "operational_storage"}}
PREPROCESSING_POLICY_HASH = sha256_bytes(canonical_json_bytes(policy_body))
require(PREPROCESSING_POLICY_HASH == str(PREPROCESSING_POLICY.get("preprocessing_policy_hash", "")), "PRE-02 preprocessing policy self-hash mismatch.")

REPRESENTATION, REPRESENTATION_HASH = verify_self_hashed_json(REPRESENTATION_PATH, "transformed_representation_manifest_hash")
CORRUPTION, CORRUPTION_HASH = verify_self_hashed_json(CORRUPTION_PATH, "corruption_group_manifest_hash")
FITTED_PREPROCESSOR_HASH = sha256_file(FITTED_PREPROCESSOR_PATH)

require(PREPROCESSING_POLICY_HASH == PRE02_INTEGRITY.get("preprocessing_policy_hash"), "PRE-02 policy hash disagrees with integrity report.")
require(REPRESENTATION_HASH == PRE02_INTEGRITY.get("transformed_representation_manifest_hash"), "PRE-02 representation hash disagrees with integrity report.")
require(CORRUPTION_HASH == PRE02_INTEGRITY.get("corruption_group_manifest_hash"), "PRE-02 corruption hash disagrees with integrity report.")
require(FITTED_PREPROCESSOR_HASH == PRE02_INTEGRITY.get("fitted_preprocessor_hash"), "PRE-02 fitted preprocessor hash disagrees with integrity report.")
require(REPRESENTATION.get("feature_order_hash") == PRE02_INTEGRITY.get("feature_order_hash"), "PRE-02 feature-order hash mismatch.")
require(REPRESENTATION.get("corruption_group_manifest_hash") == CORRUPTION_HASH, "Representation is not bound to the loaded corruption manifest.")
require(REPRESENTATION.get("preprocessing_policy_hash") == PREPROCESSING_POLICY_HASH, "Representation is not bound to the loaded preprocessing policy.")
require(REPRESENTATION.get("fitted_preprocessor_hash") == FITTED_PREPROCESSOR_HASH, "Representation is not bound to the fitted preprocessor.")

PRE00_STATUS = PRE02_INTEGRITY.get("upstream_verification", {}).get("pre00_status")
PRE01_STATUS = PRE02_INTEGRITY.get("upstream_verification", {}).get("pre01_status")
require(PRE00_STATUS in ALLOWED_PASS_STATUSES, f"PRE-00 upstream status is not acceptable: {PRE00_STATUS}")
require(PRE01_STATUS in ALLOWED_PASS_STATUSES, f"PRE-01 upstream status is not acceptable: {PRE01_STATUS}")
UPSTREAM_HASHES = dict(PRE02_INTEGRITY.get("upstream_hashes", {}))

GLOBAL_PROTOCOL_PATH = PROJECT_ROOT / "global" / "experiment1_v2_global_protocol.json"
require(GLOBAL_PROTOCOL_PATH.exists(), f"Global protocol is missing: {GLOBAL_PROTOCOL_PATH}")
GLOBAL_PROTOCOL = read_json(GLOBAL_PROTOCOL_PATH)
_gp_body = dict(GLOBAL_PROTOCOL); _gp_stored = str(_gp_body.pop("canonical_protocol_hash", ""));
GLOBAL_PROTOCOL_HASH = sha256_bytes(canonical_json_bytes(_gp_body))
require(GLOBAL_PROTOCOL_HASH == _gp_stored == str(UPSTREAM_HASHES.get("global_protocol_hash", "")), "Global protocol canonical hash mismatch.")

# ------------------------------------------------------------
# Global Experiment 1 v2 post-PRE-02 contracts
# ------------------------------------------------------------
SEED_SCHEDULE_SUBSTANTIVE = {
    "version": "Experiment1_v2_seed_schedule_v1",
    "protocol_version": PROTOCOL_VERSION,
    "OPTUNA_SAMPLER_SEED": 2026,
    "SEARCH_SEED": 101,
    "CONFIRMATION_SEEDS": [101, 202, 303],
    "ABLATION_SEEDS": [101, 202, 303],
    "THRESHOLD_SENSITIVITY_SEED": 101,
    "SENSITIVITY_SEEDS": [101, 202, 303],
    "FINAL_SEEDS": [101, 202, 303, 404, 505],
}
# Resolve unspecified library-level optimizer/loss/scheduler defaults without inventing them.
# If the global contract already exists, its frozen values take precedence; otherwise
# the current PyTorch defaults are captured prospectively and then frozen.
_existing_model_contract_path = GLOBAL_DIR / "model_search_contract.json"
if _existing_model_contract_path.exists():
    _existing_model_contract = read_json(_existing_model_contract_path)
    _existing_fixed = _existing_model_contract.get("fixed_training", {})
    HUBER_DELTA_EFFECTIVE = float(_existing_fixed["huber_delta"])
    ADAMW_FIXED_DEFAULTS = dict(_existing_fixed["adamw_fixed_defaults"])
    SCHEDULER_FIXED_DEFAULTS = dict(_existing_fixed["reduce_lr_on_plateau_fixed_defaults"])
else:
    _huber_sig = inspect.signature(nn.HuberLoss)
    HUBER_DELTA_EFFECTIVE = float(_huber_sig.parameters["delta"].default)
    _adam_sig = inspect.signature(torch.optim.AdamW)
    ADAMW_FIXED_DEFAULTS = {}
    for _name in ["betas", "eps", "amsgrad", "maximize", "foreach", "capturable", "differentiable", "fused"]:
        if _name in _adam_sig.parameters:
            _value = _adam_sig.parameters[_name].default
            if isinstance(_value, tuple): _value = list(_value)
            ADAMW_FIXED_DEFAULTS[_name] = _value
    _sched_sig = inspect.signature(torch.optim.lr_scheduler.ReduceLROnPlateau)
    SCHEDULER_FIXED_DEFAULTS = {}
    for _name in ["threshold", "threshold_mode", "cooldown", "eps"]:
        if _name in _sched_sig.parameters:
            SCHEDULER_FIXED_DEFAULTS[_name] = _sched_sig.parameters[_name].default

MODEL_SEARCH_CONTRACT_SUBSTANTIVE = {
    "version": "Experiment1_v2_model_search_contract_v1",
    "protocol_version": PROTOCOL_VERSION,
    "framework": "PyTorch",
    "architectures": {
        "A": [32, 16],
        "B": [64, 32],
        "C": [64, 32, 16],
    },
    "latent_fraction_candidates": [0.25, 0.50],
    "learning_rate": {"distribution": "log_uniform", "low": 1e-4, "high": 1e-3},
    "batch_size_candidates": [128, 256, 512],
    "weight_decay_candidates": [0.0, 1e-6, 1e-5, 1e-4],
    "training_loss_candidates": ["MSE", "Huber"],
    "dae_corruption_search": {
        "CICIDS2017": {"numerical_masking": [0.05, 0.10, 0.15], "categorical_masking": "N/A"},
        "UNSW-NB15": {"numerical_masking": [0.05, 0.10, 0.15], "grouped_categorical_masking": [0.02, 0.05, 0.10]},
    },
    "fixed_training": {
        "optimizer": "AdamW", "hidden_activation": "ReLU", "output_activation": "Linear",
        "weight_initialization": "Kaiming_normal", "maximum_epochs": 150,
        "early_stopping_patience": 12, "lr_scheduler": "ReduceLROnPlateau",
        "lr_scheduler_patience": 5, "lr_reduction_factor": 0.5,
        "minimum_learning_rate": 1e-6, "gradient_clip_norm": 1.0,
        "huber_delta": HUBER_DELTA_EFFECTIVE,
        "huber_delta_source": "PyTorch HuberLoss default captured when contract first frozen",
        "adamw_fixed_defaults": ADAMW_FIXED_DEFAULTS,
        "reduce_lr_on_plateau_fixed_defaults": SCHEDULER_FIXED_DEFAULTS,
        "kaiming_normal_details": {"mode": "fan_in", "nonlinearity": "relu", "bias": "zero"},
    },
    "formal_unique_completed_budget_per_model_per_dataset": 20,
    "smoke_outputs_excluded_from_formal": True,
    "duplicates_do_not_count": True,
    "duplicate_trial_optuna_state": "PRUNED with duplicate_detected=true; administrative duplicate state only",
    "model_pruning_enabled": False,
    "failed_interrupted_trials_do_not_count": True,
    "search_shortlist_size": 3,
    "search_stage_ranking": ["development_MCC_desc", "FPR_asc", "PR_AUC_desc", "parameter_count_asc", "training_duration_asc"],
    "confirmation_selection_hierarchy": ["mean_development_MCC_desc", "mean_FPR_asc", "mean_PR_AUC_desc", "MCC_SD_asc", "parameter_count_asc", "mean_training_duration_asc"],
    "family_metrics_used_for_selection": False,
    "reconstruction_score": "per_observation_mean_squared_reconstruction_error",
}
_existing_threshold_contract_path = GLOBAL_DIR / "threshold_contract.json"
if _existing_threshold_contract_path.exists():
    _existing_threshold_contract = read_json(_existing_threshold_contract_path)
    QUANTILE_METHOD_EFFECTIVE = str(_existing_threshold_contract["quantile_method"])
    DECISION_OPERATOR_EFFECTIVE = str(_existing_threshold_contract["decision_operator"])
else:
    _quantile_sig = inspect.signature(np.quantile)
    QUANTILE_METHOD_EFFECTIVE = str(_quantile_sig.parameters["method"].default)
    # Prospectively frozen before any model run; later notebooks consume the global contract.
    DECISION_OPERATOR_EFFECTIVE = ">"

THRESHOLD_CONTRACT_SUBSTANTIVE = {
    "version": "Experiment1_v2_threshold_contract_v1",
    "protocol_version": PROTOCOL_VERSION,
    "calibration_partition": "benign_threshold_calibration",
    "primary_percentile": 99.0,
    "sensitivity_percentiles": [95.0, 97.0, 99.0, 99.5],
    "quantile_implementation": "numpy.quantile",
    "quantile_method": QUANTILE_METHOD_EFFECTIVE,
    "quantile_method_source": "NumPy quantile default captured when contract first frozen",
    "decision_operator": DECISION_OPERATOR_EFFECTIVE,
    "attack_if": "reconstruction_error > threshold" if DECISION_OPERATOR_EFFECTIVE == ">" else ("reconstruction_error " + DECISION_OPERATOR_EFFECTIVE + " threshold"),
}
METRIC_CONTRACT_SUBSTANTIVE = {
    "version": "Experiment1_v2_metric_contract_v1",
    "protocol_version": PROTOCOL_VERSION,
    "positive_class": "attack=1",
    "primary_metrics": ["MCC", "Recall", "F1", "Precision", "PR-AUC", "FPR", "FP_per_1000_benign", "Balanced_Accuracy"],
    "secondary_score_metrics": ["ROC-AUC", "Average_Precision", "TPR_at_1pct_FPR"],
    "PR_AUC": {"implementation": "sklearn.precision_recall_curve + sklearn.metrics.auc(recall, precision)", "distinct_from_average_precision": True},
    "Average_Precision": {"implementation": "sklearn.metrics.average_precision_score"},
    "FP_per_1000_benign": "1000 * FP / N_benign",
    "TPR_at_1pct_FPR": "maximum ROC TPR among points with FPR <= 0.01",
    "accuracy_primary": False,
}
ARTIFACT_CONTRACT_SUBSTANTIVE = {
    "version": "Experiment1_v2_artifact_contract_v1",
    "protocol_version": PROTOCOL_VERSION,
    "per_run_required": [
        "effective_config.json", "run_manifest.json", "checkpoint.pt", "training_history.csv",
        "calibration_scores.parquet", "evaluation_predictions.parquet", "threshold_manifest.json",
        "metrics.json", "environment.json"
    ],
    "observation_level_fields": ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed", "config_hash", "run_hash", "attack_family"],
    "failed_runs_preserved": True,
    "checkpoint_paths_run_unique": True,
}

CONTRACT_SPECS = {
    "seed_schedule.json": SEED_SCHEDULE_SUBSTANTIVE,
    "model_search_contract.json": MODEL_SEARCH_CONTRACT_SUBSTANTIVE,
    "threshold_contract.json": THRESHOLD_CONTRACT_SUBSTANTIVE,
    "metric_contract.json": METRIC_CONTRACT_SUBSTANTIVE,
    "artifact_contract.json": ARTIFACT_CONTRACT_SUBSTANTIVE,
}

CONTRACTS = {}
CONTRACT_HASHES = {}
for filename, substantive in CONTRACT_SPECS.items():
    path = GLOBAL_DIR / filename
    if ALLOW_GLOBAL_CONTRACT_CREATION:
        obj, h = freeze_contract(path, substantive)
    else:
        obj, h = verify_contract(path)
        body = {k: v for k, v in obj.items() if k not in {"contract_hash", "created_at_utc"}}
        require(body == substantive, f"Global contract differs from this notebook's frozen Experiment 1 v2 specification: {path}")
    CONTRACTS[filename] = obj
    CONTRACT_HASHES[filename] = h

SEED_SCHEDULE = CONTRACTS["seed_schedule.json"]
MODEL_SEARCH_CONTRACT = CONTRACTS["model_search_contract.json"]
THRESHOLD_CONTRACT = CONTRACTS["threshold_contract.json"]
METRIC_CONTRACT = CONTRACTS["metric_contract.json"]
ARTIFACT_CONTRACT = CONTRACTS["artifact_contract.json"]

UPSTREAM_IDENTITY = {
    "master_dataset_sha256": UPSTREAM_HASHES.get("master_dataset_sha256"),
    "partition_manifest_canonical_hash": UPSTREAM_HASHES.get("partition_manifest_canonical_hash"),
    "partition_manifest_file_sha256": UPSTREAM_HASHES.get("partition_manifest_file_sha256"),
    "preprocessing_policy_hash": PREPROCESSING_POLICY_HASH,
    "fitted_preprocessor_hash": FITTED_PREPROCESSOR_HASH,
    "transformed_representation_manifest_hash": REPRESENTATION_HASH,
    "feature_order_hash": REPRESENTATION.get("feature_order_hash"),
    "corruption_group_manifest_hash": CORRUPTION_HASH,
    **{f"contract::{k}": v for k, v in CONTRACT_HASHES.items()},
}

print("PRE-00 status:", PRE00_STATUS)
print("PRE-01 status:", PRE01_STATUS)
print("PRE-02 status:", PRE02_INTEGRITY.get("final_status"))
print("PASS: upstream PRE-00/PRE-01/PRE-02 identities and global model-stage contracts are frozen and consistent.")
display(pd.DataFrame([{"identity": k, "hash": v} for k, v in UPSTREAM_IDENTITY.items()]))


PRE-00 status: PASS_WITH_NONBLOCKING_WARNINGS
PRE-01 status: PASS
PRE-02 status: PASS_WITH_NONBLOCKING_WARNINGS
PASS: upstream PRE-00/PRE-01/PRE-02 identities and global model-stage contracts are frozen and consistent.


,identity,hash
0,master_dataset_sha256,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...
1,partition_manifest_canonical_hash,724eeca41efa11a8ff58cba6efd172d851e9ad43870eaf...
2,partition_manifest_file_sha256,1e37421c19afbed73c669cd63f6de580a41084a7f64a7d...
3,preprocessing_policy_hash,2a2096b1d219bc833855766af6c640acc7f5cb91e1e751...
4,fitted_preprocessor_hash,7d1b4d0e2d34596b7280c36c4db9696d3e645c57e439a7...
5,transformed_representation_manifest_hash,cb027f44316f558b39e77802d8bb6723df534554be22e6...
6,feature_order_hash,68ecdade80a69e53463509c822f6ec474831a3c4289f7b...
7,corruption_group_manifest_hash,1d751d5bc6a52f58eb59fa6f0864a088ac359825f4d605...
8,contract::seed_schedule.json,e9c16b79cfb3677ce3bba3d4a56dda32e47c290df76f03...
9,contract::model_search_contract.json,a18e5092776475ff1153b3f7982df3b6f609889fa25039...


In [6]:
# ============================================================
# 6. FROZEN TRANSFORMED-DATA ACCESS / LOCAL CACHE
# ============================================================
ALLOW_FINAL_TEST_OUTCOMES = False
EXPERIMENTAL_PARTITIONS = ["benign_weight_train", "benign_early_stop", "benign_threshold_calibration", "development", "final_test"]
FEATURE_NAMES = list(REPRESENTATION.get("exact_transformed_feature_order", []))
INPUT_DIM = int(REPRESENTATION.get("total_transformed_dimensionality", -1))
TRANSFORMED_DTYPE = str(REPRESENTATION.get("transformed_dtype"))
require(INPUT_DIM > 0 and len(FEATURE_NAMES) == INPUT_DIM, "Frozen transformed representation has invalid dimension/feature order.")
require(TRANSFORMED_DTYPE in {"float32", "float64"}, f"Unsupported frozen transformed dtype: {TRANSFORMED_DTYPE}")
NP_DTYPE = np.dtype(TRANSFORMED_DTYPE)
TORCH_DTYPE = torch.float32 if TRANSFORMED_DTYPE == "float32" else torch.float64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DATASET_NAME == "CICIDS2017":
    require(CORRUPTION.get("categorical_groups") == {}, "CICIDS corruption manifest unexpectedly contains categorical groups.")
    require(list(CORRUPTION.get("numerical_indices", [])) == list(range(INPUT_DIM)), "CICIDS numerical corruption indices do not cover all dimensions.")
else:
    require(set(CORRUPTION.get("categorical_groups", {})) == {"proto", "state", "service"}, "UNSW corruption groups must be exactly proto/state/service.")

LOCAL_CACHE_ROOT = Path("/content/experiment1_v2_model_cache") / DATASET_PREFIX / REPRESENTATION.get("feature_order_hash", "unknown")
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def _partition_paths(partition):
    require(partition in EXPERIMENTAL_PARTITIONS, f"Unknown partition: {partition}")
    if partition == "final_test" and not ALLOW_FINAL_TEST_OUTCOMES:
        hard_fail(f"Final-test feature/metadata access is forbidden before Notebook 07. Attempted partition={partition}")
    return (
        PRE02_ROOT / "transformed_partitions" / f"{partition}_transformed.parquet",
        PRE02_ROOT / "transformed_partitions" / f"{partition}_transformed_alignment.parquet",
        PRE02_ROOT / "raw_partitions" / f"{partition}_raw_metadata.parquet",
    )


def _partition_manifest_info(partition):
    info = REPRESENTATION.get("per_partition", {}).get(partition)
    require(isinstance(info, dict), f"Representation manifest lacks partition metadata: {partition}")
    return info


def verify_partition_files(partition, verify_model_bytes=True):
    model_path, alignment_path, metadata_path = _partition_paths(partition)
    info = _partition_manifest_info(partition)
    for p in [model_path, alignment_path, metadata_path]:
        require(p.exists(), f"Required PRE-02 artifact is missing: {p}")
    if verify_model_bytes:
        require(sha256_file(model_path) == info.get("transformed_artifact_sha256"), f"Transformed artifact SHA mismatch for {partition}")
    require(sha256_file(alignment_path) == info.get("transformed_alignment_sha256"), f"Alignment artifact SHA mismatch for {partition}")
    require(sha256_file(metadata_path) == info.get("raw_metadata_sha256"), f"Raw metadata SHA mismatch for {partition}")
    pf = pq.ParquetFile(model_path)
    require(pf.schema_arrow.names == FEATURE_NAMES, f"Feature order mismatch for {partition}")
    require(int(pf.metadata.num_rows) == int(info.get("transformed_row_count")), f"Row count mismatch for {partition}")
    return model_path, alignment_path, metadata_path


def ensure_local_matrix(partition):
    model_path, _, _ = verify_partition_files(partition, verify_model_bytes=True)
    info = _partition_manifest_info(partition)
    rows = int(info["transformed_row_count"])
    cache_path = LOCAL_CACHE_ROOT / f"{partition}.npy"
    meta_path = LOCAL_CACHE_ROOT / f"{partition}.cache.json"
    expected_meta = {
        "source_sha256": info["transformed_artifact_sha256"],
        "rows": rows,
        "columns": INPUT_DIM,
        "dtype": TRANSFORMED_DTYPE,
        "feature_order_hash": REPRESENTATION["feature_order_hash"],
    }
    if cache_path.exists() and meta_path.exists():
        try:
            observed = read_json(meta_path)
            mm = np.load(cache_path, mmap_mode="r")
            if observed == expected_meta and mm.shape == (rows, INPUT_DIM) and str(mm.dtype) == TRANSFORMED_DTYPE:
                return mm
        except Exception:
            pass
        cache_path.unlink(missing_ok=True); meta_path.unlink(missing_ok=True)
    mm = np.lib.format.open_memmap(cache_path, mode="w+", dtype=NP_DTYPE, shape=(rows, INPUT_DIM))
    pf = pq.ParquetFile(model_path)
    offset = 0
    for batch in pf.iter_batches(batch_size=50_000, columns=FEATURE_NAMES):
        arr = batch.to_pandas().to_numpy(dtype=NP_DTYPE, copy=False)
        require(np.isfinite(arr).all(), f"Non-finite value found while caching {partition}")
        mm[offset:offset+len(arr)] = arr
        offset += len(arr)
    require(offset == rows, f"Local cache row count mismatch for {partition}")
    mm.flush(); del mm
    atomic_write_json(meta_path, expected_meta)
    return np.load(cache_path, mmap_mode="r")


def load_partition_metadata(partition):
    _, alignment_path, metadata_path = verify_partition_files(partition, verify_model_bytes=False)
    alignment = pd.read_parquet(alignment_path)
    metadata = pd.read_parquet(metadata_path)
    require(len(alignment) == len(metadata), f"Metadata/alignment row count mismatch for {partition}")
    require(alignment["row_id"].astype(str).reset_index(drop=True).equals(metadata["row_id"].astype(str).reset_index(drop=True)), f"Metadata/alignment row_id mismatch for {partition}")
    require((alignment["partition"].astype(str) == partition).all(), f"Alignment partition label mismatch for {partition}")
    require((metadata["partition"].astype(str) == partition).all(), f"Metadata partition label mismatch for {partition}")
    require("binary_label" in metadata.columns, f"binary_label missing from metadata for {partition}")
    y = pd.to_numeric(metadata["binary_label"], errors="raise").astype(int)
    require(set(y.unique()).issubset({0,1}), f"Unexpected binary labels in {partition}: {sorted(y.unique())}")
    metadata = metadata.copy(); metadata["binary_label"] = y
    return metadata.reset_index(drop=True)


def verify_role_contracts(include_final=False):
    rows = []
    for p in ["benign_weight_train", "benign_early_stop", "benign_threshold_calibration", "development"] + (["final_test"] if include_final else []):
        meta = load_partition_metadata(p)
        attacks = int(meta["binary_label"].sum())
        if p in {"benign_weight_train", "benign_early_stop", "benign_threshold_calibration"}:
            require(attacks == 0, f"{p} is not benign-only.")
        if p in {"development", "final_test"}:
            require(meta["binary_label"].nunique() == 2, f"{p} must contain benign and attack observations.")
        rows.append({"partition": p, "rows": len(meta), "attack_rows": attacks, "benign_rows": len(meta)-attacks})
        del meta; gc.collect()
    display(pd.DataFrame(rows))
    return rows

print("Frozen input dimension:", INPUT_DIM)
print("Frozen transformed dtype:", TRANSFORMED_DTYPE)
print("Model device:", DEVICE)


Frozen input dimension: 78
Frozen transformed dtype: float32
Model device: cuda


In [7]:
# ============================================================
# 7. MODEL, TRAINING, CORRUPTION, THRESHOLD, AND METRIC UTILITIES
# ============================================================
ARCHITECTURES = MODEL_SEARCH_CONTRACT["architectures"]
FIXED_TRAINING = MODEL_SEARCH_CONTRACT["fixed_training"]
_adam_params = set(inspect.signature(torch.optim.AdamW).parameters)
for _k in FIXED_TRAINING.get("adamw_fixed_defaults", {}):
    require(_k in _adam_params, f"Frozen AdamW argument {_k!r} is unsupported by this PyTorch runtime; do not silently substitute defaults.")
_sched_params = set(inspect.signature(torch.optim.lr_scheduler.ReduceLROnPlateau).parameters)
for _k in FIXED_TRAINING.get("reduce_lr_on_plateau_fixed_defaults", {}):
    require(_k in _sched_params, f"Frozen scheduler argument {_k!r} is unsupported by this PyTorch runtime; do not silently substitute defaults.")


def set_global_seed(seed):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic algorithms are requested where available; some CUDA operations
    # may still depend on runtime/library support and this limitation is recorded.
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def latent_dim_for(architecture, latent_fraction):
    hidden = ARCHITECTURES[str(architecture)]
    final_width = int(hidden[-1])
    z = max(1, int(round(final_width * float(latent_fraction))))
    require(z < INPUT_DIM, f"Latent dimension must be undercomplete: z={z}, p={INPUT_DIM}")
    return z


class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_widths, latent_dim):
        super().__init__()
        enc = []
        prev = int(input_dim)
        for width in hidden_widths:
            layer = nn.Linear(prev, int(width)); nn.init.kaiming_normal_(layer.weight, mode="fan_in", nonlinearity="relu"); nn.init.zeros_(layer.bias)
            enc += [layer, nn.ReLU()]; prev = int(width)
        latent = nn.Linear(prev, int(latent_dim)); nn.init.kaiming_normal_(latent.weight, mode="fan_in", nonlinearity="relu"); nn.init.zeros_(latent.bias)
        enc += [latent]
        self.encoder = nn.Sequential(*enc)

        dec = []
        prev = int(latent_dim)
        for width in reversed(hidden_widths):
            layer = nn.Linear(prev, int(width)); nn.init.kaiming_normal_(layer.weight, mode="fan_in", nonlinearity="relu"); nn.init.zeros_(layer.bias)
            dec += [layer, nn.ReLU()]; prev = int(width)
        out = nn.Linear(prev, int(input_dim)); nn.init.kaiming_normal_(out.weight, mode="fan_in", nonlinearity="relu"); nn.init.zeros_(out.bias)
        dec += [out]
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def build_model(config):
    return Autoencoder(INPUT_DIM, ARCHITECTURES[config["architecture"]], int(config["latent_dim"])).to(device=DEVICE, dtype=TORCH_DTYPE)


def parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def apply_structured_corruption(clean_x, config):
    x = clean_x.clone()
    numerical_p = float(config.get("numerical_masking", 0.0) or 0.0)
    numerical_indices = list(map(int, CORRUPTION.get("numerical_indices", [])))
    if numerical_p > 0 and numerical_indices:
        idx = torch.tensor(numerical_indices, dtype=torch.long, device=x.device)
        mask = torch.rand((x.shape[0], len(numerical_indices)), device=x.device) < numerical_p
        vals = x.index_select(1, idx)
        vals = torch.where(mask, torch.zeros_like(vals), vals)
        x[:, idx] = vals
    cat_p = float(config.get("grouped_categorical_masking", 0.0) or 0.0)
    if cat_p > 0:
        groups = CORRUPTION.get("categorical_groups", {})
        require(bool(groups), "Categorical masking requested but corruption manifest contains no categorical groups.")
        for _, group in groups.items():
            idx = torch.tensor(list(map(int, group)), dtype=torch.long, device=x.device)
            group_mask = (torch.rand((x.shape[0], 1), device=x.device) < cat_p)
            vals = x.index_select(1, idx)
            vals = torch.where(group_mask, torch.zeros_like(vals), vals)
            x[:, idx] = vals
    return x


def training_loss_fn(name):
    if name == "MSE": return nn.MSELoss(reduction="mean")
    if name == "Huber": return nn.HuberLoss(delta=float(FIXED_TRAINING["huber_delta"]), reduction="mean")
    hard_fail(f"Unsupported training loss: {name}")


def iter_sequential_batches(mm, batch_size):
    for start in range(0, len(mm), int(batch_size)):
        arr = np.asarray(mm[start:start+int(batch_size)])
        yield torch.tensor(arr, dtype=TORCH_DTYPE, device=DEVICE)


def clean_validation_loss(model, mm, batch_size, criterion):
    model.eval(); total = 0.0; n = 0
    with torch.no_grad():
        for xb in iter_sequential_batches(mm, batch_size):
            pred = model(xb); loss = criterion(pred, xb)
            total += float(loss.item()) * len(xb); n += len(xb)
    return total / max(n, 1)


def train_model(config, model_type, seed, checkpoint_path):
    set_global_seed(seed)
    train_mm = ensure_local_matrix("benign_weight_train")
    early_mm = ensure_local_matrix("benign_early_stop")
    model = build_model(config)
    pcount = parameter_count(model)
    _adam_kwargs = dict(FIXED_TRAINING.get("adamw_fixed_defaults", {}))
    if "betas" in _adam_kwargs: _adam_kwargs["betas"] = tuple(_adam_kwargs["betas"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config["learning_rate"]), weight_decay=float(config["weight_decay"]), **_adam_kwargs)
    _sched_kwargs = dict(FIXED_TRAINING.get("reduce_lr_on_plateau_fixed_defaults", {}))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=float(FIXED_TRAINING["lr_reduction_factor"]),
        patience=int(FIXED_TRAINING["lr_scheduler_patience"]), min_lr=float(FIXED_TRAINING["minimum_learning_rate"]), **_sched_kwargs
    )
    criterion = training_loss_fn(config["training_loss"])
    rng = np.random.default_rng(int(seed))
    best_val = float("inf"); best_epoch = -1; stale = 0; history = []
    started = time.perf_counter()
    checkpoint_path = Path(checkpoint_path); checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, int(FIXED_TRAINING["maximum_epochs"]) + 1):
        model.train(); order = rng.permutation(len(train_mm)); total = 0.0; n = 0
        bs = int(config["batch_size"])
        for start in range(0, len(order), bs):
            idx = order[start:start+bs]
            clean = torch.tensor(np.asarray(train_mm[idx]), dtype=TORCH_DTYPE, device=DEVICE)
            inp = apply_structured_corruption(clean, config) if model_type == "DAE" else clean
            optimizer.zero_grad(set_to_none=True)
            pred = model(inp); loss = criterion(pred, clean); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(FIXED_TRAINING["gradient_clip_norm"]))
            optimizer.step()
            total += float(loss.item()) * len(clean); n += len(clean)
        train_loss = total / max(n,1)
        val_loss = clean_validation_loss(model, early_mm, bs, criterion)
        scheduler.step(val_loss)
        lr_now = float(optimizer.param_groups[0]["lr"])
        history.append({"epoch": epoch, "training_loss": train_loss, "early_stop_validation_loss": val_loss, "learning_rate": lr_now})
        if val_loss < best_val:
            best_val = val_loss; best_epoch = epoch; stale = 0
            torch.save({
                "state_dict": model.state_dict(), "config": config, "model_type": model_type,
                "seed": int(seed), "best_epoch": int(best_epoch), "best_validation_loss": float(best_val),
                "input_dim": INPUT_DIM, "feature_order_hash": REPRESENTATION["feature_order_hash"],
            }, checkpoint_path)
        else:
            stale += 1
            if stale >= int(FIXED_TRAINING["early_stopping_patience"]):
                break
    duration = float(time.perf_counter() - started)
    require(checkpoint_path.exists(), "Training completed without a best checkpoint.")
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    return model, pd.DataFrame(history), int(best_epoch), float(best_val), duration, pcount


def reconstruction_errors(model, mm, batch_size):
    model.eval(); parts = []
    with torch.no_grad():
        for xb in iter_sequential_batches(mm, batch_size):
            pred = model(xb)
            err = torch.mean((xb - pred) ** 2, dim=1)
            parts.append(err.detach().cpu().numpy().astype(np.float64, copy=False))
    return np.concatenate(parts) if parts else np.empty(0, dtype=np.float64)


def quantile_threshold(scores, percentile):
    return float(np.quantile(np.asarray(scores, dtype=np.float64), float(percentile)/100.0, method=THRESHOLD_CONTRACT["quantile_method"]))


def predict_from_threshold(scores, threshold):
    require(THRESHOLD_CONTRACT["decision_operator"] == ">", "This implementation expects the frozen '>' decision operator.")
    return (np.asarray(scores) > float(threshold)).astype(np.int8)


def compute_metrics(y_true, scores, threshold):
    y = np.asarray(y_true, dtype=int); scores = np.asarray(scores, dtype=np.float64)
    pred = predict_from_threshold(scores, threshold)
    require(set(np.unique(y)).issubset({0,1}), "Metrics received non-binary labels.")
    tp = int(np.sum((y==1)&(pred==1))); tn = int(np.sum((y==0)&(pred==0)))
    fp = int(np.sum((y==0)&(pred==1))); fn = int(np.sum((y==1)&(pred==0)))
    n_benign = int(np.sum(y==0))
    fpr = fp / n_benign if n_benign else float("nan")
    precision_curve, recall_curve, _ = precision_recall_curve(y, scores)
    pr_auc = float(auc(recall_curve, precision_curve))
    roc_auc = float(roc_auc_score(y, scores)) if len(np.unique(y)) == 2 else float("nan")
    ap = float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else float("nan")
    if len(np.unique(y)) == 2:
        fpr_curve, tpr_curve, _ = roc_curve(y, scores)
        eligible = tpr_curve[fpr_curve <= 0.01]
        tpr1 = float(np.max(eligible)) if len(eligible) else 0.0
    else:
        tpr1 = float("nan")
    return {
        "MCC": float(matthews_corrcoef(y, pred)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "F1": float(f1_score(y, pred, zero_division=0)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "PR-AUC": pr_auc,
        "FPR": float(fpr),
        "FP_per_1000_benign": float(1000.0 * fp / n_benign) if n_benign else float("nan"),
        "Balanced_Accuracy": float(balanced_accuracy_score(y, pred)),
        "ROC-AUC": roc_auc,
        "Average_Precision": ap,
        "TPR_at_1pct_FPR": tpr1,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "threshold": float(threshold),
    }


def family_diagnostics(metadata, predictions):
    df = pd.DataFrame({
        "y_true": metadata["binary_label"].astype(int).to_numpy(),
        "y_pred": np.asarray(predictions, dtype=int),
        "attack_family": metadata.get("attack_family", pd.Series([None]*len(metadata))).astype("string").to_numpy(),
    })
    attacks = df[df["y_true"] == 1].copy()
    require(len(attacks) > 0, "No attacks are available for family diagnostics.")
    require(not pd.isna(attacks["attack_family"]).any(), "Attack-family metadata is missing for one or more attack observations.")
    total_attack = len(attacks); total_tp = int((attacks["y_pred"]==1).sum()); total_fn = int((attacks["y_pred"]==0).sum())
    rows = []
    for family, g in attacks.groupby("attack_family", sort=True, dropna=False):
        support = len(g); tp = int((g["y_pred"]==1).sum()); fn = int((g["y_pred"]==0).sum())
        rows.append({
            "attack_family": str(family), "support": support, "TP": tp, "FN": fn,
            "recall": float(tp/support) if support else float("nan"),
            "attack_share": float(support/total_attack),
            "detected_share": float(tp/total_tp) if total_tp else float("nan"),
            "missed_share": float(fn/total_fn) if total_fn else float("nan"),
        })
    return pd.DataFrame(rows)

print("PASS: model/training/threshold/metric utilities loaded. Anomaly score is fixed per-observation reconstruction MSE.")


PASS: model/training/threshold/metric utilities loaded. Anomaly score is fixed per-observation reconstruction MSE.


In [8]:
# ============================================================
# 8. CONFIG HASH, RUN HASH, RUN PERSISTENCE, EXECUTION
# ============================================================
def normalized_effective_config(model_type, params):
    c = {
        "model_type": str(model_type),
        "architecture": str(params["architecture"]),
        "latent_fraction": float(params["latent_fraction"]),
        "latent_dim": int(params.get("latent_dim", latent_dim_for(params["architecture"], params["latent_fraction"]))),
        "learning_rate": float(params["learning_rate"]),
        "batch_size": int(params["batch_size"]),
        "weight_decay": float(params["weight_decay"]),
        "training_loss": str(params["training_loss"]),
        "optimizer": "AdamW",
        "hidden_activation": "ReLU",
        "output_activation": "Linear",
        "weight_initialization": "Kaiming_normal",
        "maximum_epochs": int(FIXED_TRAINING["maximum_epochs"]),
        "early_stopping_patience": int(FIXED_TRAINING["early_stopping_patience"]),
        "lr_scheduler_patience": int(FIXED_TRAINING["lr_scheduler_patience"]),
        "lr_reduction_factor": float(FIXED_TRAINING["lr_reduction_factor"]),
        "minimum_learning_rate": float(FIXED_TRAINING["minimum_learning_rate"]),
        "gradient_clip_norm": float(FIXED_TRAINING["gradient_clip_norm"]),
        "huber_delta": float(FIXED_TRAINING["huber_delta"]),
        "adamw_fixed_defaults": FIXED_TRAINING.get("adamw_fixed_defaults", {}),
        "reduce_lr_on_plateau_fixed_defaults": FIXED_TRAINING.get("reduce_lr_on_plateau_fixed_defaults", {}),
        "kaiming_normal_details": FIXED_TRAINING.get("kaiming_normal_details", {}),
        "numerical_masking": float(params.get("numerical_masking", 0.0) or 0.0),
        "corruption_group_manifest_hash": CORRUPTION_HASH,
    }
    if DATASET_NAME == "UNSW-NB15" and model_type in {"DAE", "AE_MATCHED"}:
        c["grouped_categorical_masking"] = float(params.get("grouped_categorical_masking", 0.0) or 0.0)
    return c


def config_hash(config):
    return sha256_bytes(canonical_json_bytes(config))


def make_run_hash(stage, model_type, config_h, seed, attempt=1):
    payload = {
        "dataset": DATASET_NAME, "stage": stage, "model_type": model_type,
        "config_hash": config_h, "seed": int(seed),
        "partition_manifest_hash": UPSTREAM_HASHES.get("partition_manifest_canonical_hash"),
        "preprocessing_manifest_hash": PREPROCESSING_POLICY_HASH,
        "transformed_representation_hash": REPRESENTATION_HASH,
        "corruption_group_manifest_hash": CORRUPTION_HASH,
        "code_version": CODE_VERSION, "notebook_version": NOTEBOOK_VERSION,
        "attempt": int(attempt),
    }
    return sha256_bytes(canonical_json_bytes(payload))


def environment_for_run():
    return {**ENVIRONMENT_INFO, "device": str(DEVICE), "torch_dtype": str(TORCH_DTYPE), "code_version": CODE_VERSION, "notebook_version": NOTEBOOK_VERSION}


def _find_existing_complete_run(stage_root, stage, model_type, cfg_hash, seed):
    runs_dir = Path(stage_root) / "runs"
    if not runs_dir.exists(): return None
    for p in runs_dir.glob("*/run_manifest.json"):
        try:
            obj, _ = verify_self_hashed_json(p, "run_manifest_hash")
            if (obj.get("completion_status") == "COMPLETE" and obj.get("stage") == stage and obj.get("model_type") == model_type
                and obj.get("config_hash") == cfg_hash and int(obj.get("seed")) == int(seed)
                and obj.get("upstream_identity") == UPSTREAM_IDENTITY):
                return p.parent, obj
        except Exception:
            continue
    return None


def execute_training_run(stage_root, stage, model_type, params, seed, evaluation_partition="development", allow_reuse=True):
    require(evaluation_partition in {"development", "final_test"}, "Training-run evaluation partition must be development or final_test.")
    if evaluation_partition == "final_test": require(ALLOW_FINAL_TEST_OUTCOMES, "Final-test outcomes are sealed before Notebook 07.")
    config = normalized_effective_config(model_type, params)
    cfg_hash = config_hash(config)
    if allow_reuse:
        found = _find_existing_complete_run(stage_root, stage, model_type, cfg_hash, seed)
        if found is not None:
            run_dir, manifest = found
            print("Reusing validated complete run:", run_dir.name)
            return manifest

    runs_dir = Path(stage_root) / "runs"; runs_dir.mkdir(parents=True, exist_ok=True)
    attempts = []
    for pattern in ["*/run_manifest.json", "*/failure_manifest.json"]:
        for p in runs_dir.glob(pattern):
            try:
                obj = read_json(p)
                if obj.get("stage") == stage and obj.get("model_type") == model_type and obj.get("config_hash") == cfg_hash and int(obj.get("seed", -1)) == int(seed):
                    attempts.append(int(obj.get("attempt", 1)))
            except Exception:
                pass
    attempt = max(attempts, default=0) + 1
    r_hash = make_run_hash(stage, model_type, cfg_hash, seed, attempt)
    run_dir = runs_dir / r_hash
    require(not run_dir.exists(), f"Run directory unexpectedly exists: {run_dir}")
    run_dir.mkdir(parents=True)
    checkpoint_path = run_dir / "checkpoint.pt"
    atomic_write_json(run_dir / "effective_config.json", {"config_hash": cfg_hash, "effective_config": config})
    atomic_write_json(run_dir / "environment.json", environment_for_run())

    try:
        model, history, best_epoch, best_val, duration, pcount = train_model(config, "DAE" if model_type == "DAE" else "AE", seed, checkpoint_path)
        atomic_write_csv(run_dir / "training_history.csv", history)
        calibration_mm = ensure_local_matrix("benign_threshold_calibration")
        calibration_scores = reconstruction_errors(model, calibration_mm, config["batch_size"])
        calibration_meta = load_partition_metadata("benign_threshold_calibration")
        require(len(calibration_scores) == len(calibration_meta), "Calibration score/metadata length mismatch.")
        cal_df = pd.DataFrame({"row_id": calibration_meta["row_id"].astype(str), "reconstruction_error": calibration_scores})
        atomic_write_parquet(run_dir / "calibration_scores.parquet", cal_df)
        threshold = quantile_threshold(calibration_scores, THRESHOLD_CONTRACT["primary_percentile"])
        threshold_substantive = {
            "percentile": float(THRESHOLD_CONTRACT["primary_percentile"]), "threshold": threshold,
            "calibration_partition": "benign_threshold_calibration",
            "calibration_partition_hash": REPRESENTATION["per_partition"]["benign_threshold_calibration"]["transformed_artifact_sha256"],
            "calibration_observations": int(len(calibration_scores)),
            "quantile_implementation": THRESHOLD_CONTRACT["quantile_implementation"], "quantile_method": THRESHOLD_CONTRACT["quantile_method"],
            "score_definition": MODEL_SEARCH_CONTRACT["reconstruction_score"], "decision_operator": THRESHOLD_CONTRACT["decision_operator"],
        }
        threshold_manifest = {**threshold_substantive, "threshold_manifest_hash": sha256_bytes(canonical_json_bytes(threshold_substantive))}
        atomic_write_json(run_dir / "threshold_manifest.json", threshold_manifest)

        eval_mm = ensure_local_matrix(evaluation_partition)
        eval_scores = reconstruction_errors(model, eval_mm, config["batch_size"])
        eval_meta = load_partition_metadata(evaluation_partition)
        require(len(eval_scores) == len(eval_meta), "Evaluation score/metadata length mismatch.")
        y_true = eval_meta["binary_label"].astype(int).to_numpy()
        metrics = compute_metrics(y_true, eval_scores, threshold)
        preds = predict_from_threshold(eval_scores, threshold)
        out = pd.DataFrame({
            "row_id": eval_meta["row_id"].astype(str), "y_true": y_true,
            "reconstruction_error": eval_scores, "threshold": float(threshold), "y_pred": preds,
            "model": model_type, "seed": int(seed), "config_hash": cfg_hash, "run_hash": r_hash,
            "attack_family": eval_meta.get("attack_family", pd.Series([None]*len(eval_meta))).astype("string"),
        })
        if "feature_fingerprint" in eval_meta.columns:
            out["feature_fingerprint"] = eval_meta["feature_fingerprint"].astype("string")
        atomic_write_parquet(run_dir / "evaluation_predictions.parquet", out)
        atomic_write_json(run_dir / "metrics.json", metrics)
        checkpoint_hash = sha256_file(checkpoint_path)
        run_substantive = {
            "dataset": DATASET_NAME, "stage": stage, "model_type": model_type, "seed": int(seed),
            "attempt": int(attempt), "effective_config": config, "config_hash": cfg_hash, "run_hash": r_hash,
            "upstream_identity": UPSTREAM_IDENTITY, "evaluation_partition": evaluation_partition,
            "checkpoint_path": str(checkpoint_path), "checkpoint_sha256": checkpoint_hash,
            "best_epoch": int(best_epoch), "best_early_stop_loss": float(best_val),
            "training_duration_seconds": duration, "parameter_count": pcount,
            "threshold": float(threshold), "metrics": metrics,
            "artifact_hashes": {
                "training_history": sha256_file(run_dir / "training_history.csv"),
                "calibration_scores": sha256_file(run_dir / "calibration_scores.parquet"),
                "evaluation_predictions": sha256_file(run_dir / "evaluation_predictions.parquet"),
                "threshold_manifest": sha256_file(run_dir / "threshold_manifest.json"),
                "metrics": sha256_file(run_dir / "metrics.json"),
                "environment": sha256_file(run_dir / "environment.json"),
            },
            "completion_status": "COMPLETE",
        }
        manifest = {**run_substantive, "created_at_utc": datetime.now(timezone.utc).isoformat(), "run_manifest_hash": sha256_bytes(canonical_json_bytes(run_substantive))}
        atomic_write_json(run_dir / "run_manifest.json", manifest)
        return manifest
    except Exception as exc:
        failure = {
            "dataset": DATASET_NAME, "stage": stage, "model_type": model_type, "seed": int(seed), "attempt": int(attempt),
            "config_hash": cfg_hash, "run_hash": r_hash, "effective_config": config,
            "failure_type": type(exc).__name__, "failure_message": str(exc), "traceback": traceback.format_exc(),
            "created_at_utc": datetime.now(timezone.utc).isoformat(), "completion_status": "FAILED",
        }
        atomic_write_json(run_dir / "failure_manifest.json", failure)
        raise


def collect_complete_run_manifests(stage_root, stage=None, model_type=None):
    rows = []
    for p in (Path(stage_root) / "runs").glob("*/run_manifest.json") if (Path(stage_root)/"runs").exists() else []:
        try:
            obj, _ = verify_self_hashed_json(p, "run_manifest_hash")
            if obj.get("completion_status") != "COMPLETE" or obj.get("upstream_identity") != UPSTREAM_IDENTITY: continue
            if stage is not None and obj.get("stage") != stage: continue
            if model_type is not None and obj.get("model_type") != model_type: continue
            rows.append(obj)
        except Exception:
            continue
    return rows


def run_manifest_to_row(m):
    row = {
        "model_type": m["model_type"], "seed": int(m["seed"]), "config_hash": m["config_hash"], "run_hash": m["run_hash"],
        "parameter_count": int(m["parameter_count"]), "training_duration_seconds": float(m["training_duration_seconds"]),
        "best_epoch": int(m["best_epoch"]), "threshold": float(m["threshold"]),
    }
    row.update(m["metrics"])
    return row


In [9]:
# ============================================================
# 9. CURRENT STAGE OUTPUT ROOT / ROLE GATE
# ============================================================
require(RUN_MODE in {"formal", "smoke"}, "RUN_MODE must be 'formal' or 'smoke'.")
STAGE_SLUG = 'notebook02_dae_search'
STAGE_ROOT = MODELLING_ROOT / STAGE_SLUG / "v1_0_0" / RUN_MODE
STAGE_ROOT.mkdir(parents=True, exist_ok=True)
atomic_write_json(STAGE_ROOT / "environment_stage.json", ENVIRONMENT_INFO)
verify_role_contracts(include_final=False)
print("STAGE_ROOT:", STAGE_ROOT)


,partition,rows,attack_rows,benign_rows
0,benign_weight_train,1136548,0,1136548
1,benign_early_stop,227309,0,227309
2,benign_threshold_calibration,340964,0,340964
3,development,445703,218394,227309


STAGE_ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/modelling/notebook02_dae_search/v1_0_0/formal


In [10]:
# ============================================================
# 10. NOTEBOOK 02 — DAE JOINT RANDOM SEARCH
# ============================================================

# Strict sequential execution: formal AE search must already be complete.
AE_ROOT, AE_STAGE = resolve_dependency_stage("notebook01_ae_search", require_formal=True)
require(int(AE_STAGE.get("unique_completed_configurations", 0)) == int(MODEL_SEARCH_CONTRACT["formal_unique_completed_budget_per_model_per_dataset"]), "Notebook 01 formal AE search is incomplete.")

SEARCH_MODEL_TYPE = 'DAE'
TARGET_UNIQUE_COMPLETED = int(MODEL_SEARCH_CONTRACT["formal_unique_completed_budget_per_model_per_dataset"]) if RUN_MODE == "formal" else int(SMOKE_UNIQUE_COMPLETED)
SEARCH_SEED = int(SEED_SCHEDULE["SEARCH_SEED"])
SAMPLER_SEED = int(SEED_SCHEDULE["OPTUNA_SAMPLER_SEED"])
STUDY_PATH = STAGE_ROOT / "optuna_search.sqlite3"
storage = optuna.storages.RDBStorage(url=f"sqlite:///{STUDY_PATH}", engine_kwargs={"connect_args": {"timeout": 60}})
study = optuna.create_study(study_name=f"Experiment1_v2_{DATASET_PREFIX}_{SEARCH_MODEL_TYPE}_{RUN_MODE}", direction="maximize", sampler=optuna.samplers.RandomSampler(seed=SAMPLER_SEED), storage=storage, load_if_exists=True)


def sample_params(trial):
    params = {
        "architecture": trial.suggest_categorical("architecture", list(MODEL_SEARCH_CONTRACT["architectures"])),
        "latent_fraction": trial.suggest_categorical("latent_fraction", MODEL_SEARCH_CONTRACT["latent_fraction_candidates"]),
        "learning_rate": trial.suggest_float("learning_rate", MODEL_SEARCH_CONTRACT["learning_rate"]["low"], MODEL_SEARCH_CONTRACT["learning_rate"]["high"], log=True),
        "batch_size": trial.suggest_categorical("batch_size", MODEL_SEARCH_CONTRACT["batch_size_candidates"]),
        "weight_decay": trial.suggest_categorical("weight_decay", MODEL_SEARCH_CONTRACT["weight_decay_candidates"]),
        "training_loss": trial.suggest_categorical("training_loss", MODEL_SEARCH_CONTRACT["training_loss_candidates"]),
    }
    params["latent_dim"] = latent_dim_for(params["architecture"], params["latent_fraction"])

    params["numerical_masking"] = trial.suggest_categorical("numerical_masking", MODEL_SEARCH_CONTRACT["dae_corruption_search"][DATASET_NAME]["numerical_masking"])

    return params


def completed_unique():
    runs = collect_complete_run_manifests(STAGE_ROOT, NOTEBOOK_STAGE, SEARCH_MODEL_TYPE)
    by_hash = {m["config_hash"]: m for m in runs}
    return by_hash

completed = completed_unique()
attempts_this_execution = 0
consecutive_failures = 0
while len(completed) < TARGET_UNIQUE_COMPLETED:
    require(attempts_this_execution < int(MAX_SEARCH_ATTEMPTS), f"Search attempt safety limit reached before {TARGET_UNIQUE_COMPLETED} unique completed configs. Completed={len(completed)}")
    attempts_this_execution += 1
    trial = study.ask()
    params = sample_params(trial)
    effective = normalized_effective_config(SEARCH_MODEL_TYPE, params)
    cfg_hash = config_hash(effective)
    trial.set_user_attr("config_hash", cfg_hash)
    trial.set_user_attr("run_mode", RUN_MODE)
    if cfg_hash in completed:
        trial.set_user_attr("duplicate_detected", True)
        trial.set_user_attr("duplicate_reason", "effective config_hash already completed")
        study.tell(trial, state=TrialState.PRUNED)
        continue
    try:
        manifest = execute_training_run(STAGE_ROOT, NOTEBOOK_STAGE, SEARCH_MODEL_TYPE, params, SEARCH_SEED, evaluation_partition="development", allow_reuse=True)
        trial.set_user_attr("run_hash", manifest["run_hash"])
        trial.set_user_attr("completion_status", "COMPLETE")
        study.tell(trial, float(manifest["metrics"]["MCC"]))
        completed[manifest["config_hash"]] = manifest
        consecutive_failures = 0
        print(f"Completed unique configs: {len(completed)} / {TARGET_UNIQUE_COMPLETED}")
    except Exception as exc:
        trial.set_user_attr("completion_status", "FAILED")
        trial.set_user_attr("failure", str(exc))
        study.tell(trial, state=TrialState.FAIL)
        consecutive_failures += 1
        print("Formal trial failed and remains in audit trail:", type(exc).__name__, exc)
        require(consecutive_failures < 5, "Five consecutive training failures indicate a systemic problem; stopping rather than consuming unlimited attempts.")

runs = list(completed.values())
rows = pd.DataFrame([run_manifest_to_row(m) for m in runs])
# Search-stage ranking has one common seed, so seed variability is not yet available.
ranked = rows.sort_values(
    by=["MCC", "FPR", "PR-AUC", "parameter_count", "training_duration_seconds", "config_hash"],
    ascending=[False, True, False, True, True, True], kind="mergesort"
).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked)+1))
atomic_write_csv(STAGE_ROOT / "ranked_trials.csv", ranked)

config_by_hash = {m["config_hash"]: m["effective_config"] for m in runs}
top_n = int(MODEL_SEARCH_CONTRACT["search_shortlist_size"])
top_hashes = ranked.head(top_n)["config_hash"].tolist()
top_three = [{"rank": i+1, "config_hash": h, "effective_config": config_by_hash[h]} for i, h in enumerate(top_hashes)]
atomic_write_json(STAGE_ROOT / "top_three_configs.json", {"dataset": DATASET_NAME, "model_type": SEARCH_MODEL_TYPE, "configs": top_three})

trial_rows = []
for t in study.trials:
    trial_rows.append({
        "trial_number": t.number, "state": t.state.name, "value": t.value,
        "config_hash": t.user_attrs.get("config_hash"), "run_hash": t.user_attrs.get("run_hash"),
        "duplicate_detected": t.user_attrs.get("duplicate_detected", False), "failure": t.user_attrs.get("failure"),
        **{f"param::{k}": v for k, v in t.params.items()},
    })
atomic_write_csv(STAGE_ROOT / "trial_register.csv", pd.DataFrame(trial_rows))

if RUN_MODE == "formal":
    require(len(completed) == int(MODEL_SEARCH_CONTRACT["formal_unique_completed_budget_per_model_per_dataset"]), "Formal search completion gate failed.")
    final_status = "PASS"
else:
    final_status = "PASS_WITH_NONBLOCKING_WARNINGS"

key_artifact_hashes = {
    "ranked_trials.csv": sha256_file(STAGE_ROOT / "ranked_trials.csv"),
    "top_three_configs.json": sha256_file(STAGE_ROOT / "top_three_configs.json"),
    "trial_register.csv": sha256_file(STAGE_ROOT / "trial_register.csv"),
}
stage_substantive = {
    "dataset": DATASET_NAME, "stage": NOTEBOOK_STAGE, "notebook_version": NOTEBOOK_VERSION, "code_version": CODE_VERSION,
    "run_mode": RUN_MODE, "model_type": SEARCH_MODEL_TYPE, "upstream_identity": UPSTREAM_IDENTITY,
    "unique_completed_configurations": int(len(completed)), "target_unique_completed": int(TARGET_UNIQUE_COMPLETED),
    "search_seed": SEARCH_SEED, "optuna_sampler_seed": SAMPLER_SEED,
    "top_three_config_hashes": top_hashes, "family_metrics_used_for_ranking": False,
    "key_artifact_hashes": key_artifact_hashes,
    "final_status": final_status,
    "next_authorized_stage": "Notebook 02 DAE search" if NOTEBOOK_STAGE_NUMBER == 1 and RUN_MODE == "formal" else ("Notebook 03 multi-seed confirmation" if RUN_MODE == "formal" else "NONE — smoke test only"),
}
STAGE_MANIFEST = write_stage_manifest(STAGE_ROOT / "stage_manifest.json", stage_substantive)
create_artifact_inventory(STAGE_ROOT, STAGE_ROOT / "artifact_inventory.csv")
display(ranked.head(20))
print("SEARCH STAGE STATUS:", STAGE_MANIFEST["final_status"])
print("Unique completed configurations:", len(completed))
print("Family metrics used for model selection: NO")


[I 2026-09-15 20:36:10,701] A new study created in RDB with name: Experiment1_v2_CICIDS2017_DAE_formal


Completed unique configs: 1 / 20
Completed unique configs: 2 / 20
Completed unique configs: 3 / 20
Completed unique configs: 4 / 20
Completed unique configs: 5 / 20
Completed unique configs: 6 / 20
Completed unique configs: 7 / 20
Completed unique configs: 8 / 20
Completed unique configs: 9 / 20
Completed unique configs: 10 / 20
Completed unique configs: 11 / 20
Completed unique configs: 12 / 20
Completed unique configs: 13 / 20
Completed unique configs: 14 / 20
Completed unique configs: 15 / 20
Completed unique configs: 16 / 20
Completed unique configs: 17 / 20
Completed unique configs: 18 / 20
Completed unique configs: 19 / 20
Completed unique configs: 20 / 20


,rank,model_type,seed,config_hash,run_hash,parameter_count,training_duration_seconds,best_epoch,threshold,MCC,...,FPR,FP_per_1000_benign,Balanced_Accuracy,ROC-AUC,Average_Precision,TPR_at_1pct_FPR,TP,TN,FP,FN
0,1,DAE,101,16838232aebb6211bb93d1c698985b35164410bd4d081d...,88239abf4202bf3ebdd58fd86c71c070e060bfe00842e7...,15390,1214.568547,73,0.031381,0.523962,...,0.009920,9.920417,0.718519,0.857737,0.868865,0.447228,97613,225054,2255,120781
1,2,DAE,101,0a0f3619b4715dc2291fe829dc12273433bd3969c7aaf8...,7e0cd6975746753e95b0469e6b3935bae2c9c8e83f5afa...,15390,657.877845,77,0.064504,0.523319,...,0.009846,9.845629,0.718073,0.869216,0.879750,0.446075,97402,225071,2238,120992
2,3,DAE,101,a2429d19857802165d8eb263f9d34ed01e884856f587cd...,f8d5e46aa25c60f84017dcb8ef59680d5bd66c7762fcd1...,15670,1078.865036,54,0.147472,0.522022,...,0.009876,9.876424,0.717270,0.860407,0.869643,0.444449,97058,225064,2245,121336
3,4,DAE,101,4a73f900ac86d8747268f2909c47528abe67946c3309c9...,f9eccd7d80843796067abc83caeb3699a6eaf42901f5f6...,15390,2135.856015,64,0.093999,0.520193,...,0.009832,9.832431,0.716093,0.875935,0.882587,0.442109,96534,225074,2235,121860
4,5,DAE,101,587452b11803b2f6ff7e9110cb9a4dbd8d572ce0615b88...,0b79ab33564ffbc93489635dd53ec6c445cdd852f8466b...,15390,869.222439,19,0.078082,0.517775,...,0.009736,9.735646,0.714515,0.832983,0.842774,0.438936,95824,225096,2213,122570
5,6,DAE,101,38f8c8987a1524fc3dccbf91915854e7e11319b76309c9...,945901ab0867e4c4e0964f0879ab7eb896434fa04f7de7...,15390,301.608482,9,0.139873,0.514037,...,0.009643,9.643261,0.712110,0.872570,0.880100,0.434060,94753,225117,2192,123641
6,7,DAE,101,586e1369bdd1a108f9c64aa4c5e0f57ed3d811cd44e4d5...,f20418cb5ed89bdd4ad8a369f10351687d2d7cbfb9d2c4...,14870,2000.099557,59,0.078994,0.514019,...,0.009586,9.586070,0.712067,0.828050,0.839492,0.443048,94722,225130,2179,123672
7,8,DAE,101,8d1c9e117aca4ba0a2ac0556d5604354a31847c96c670a...,48ed09f863fd1b9d7cf3b0ece19f4a65d01957e2e2bfaa...,15670,610.153706,25,0.157497,0.512885,...,0.009806,9.806035,0.711472,0.849063,0.862573,0.433405,94510,225080,2229,123884
8,9,DAE,101,dab7897cda380d2fa859882b1a0d7f3f3d586f47f3fe0a...,aff30a7539d541b145013880cca5b2ca2da7945d50fda8...,15390,1102.873116,142,0.050539,0.512801,...,0.009731,9.731247,0.711379,0.845086,0.854988,0.434119,94453,225097,2212,123941
9,10,DAE,101,1af31a0ab4ca8811f2c9292ef606ebde3be8c8d8cb866c...,dbbdc7ca038f88412bf584660c49de10c91f04be1875f9...,14870,3014.034539,95,0.146443,0.512376,...,0.009736,9.735646,0.711113,0.846857,0.854736,0.432173,94338,225096,2213,124056


SEARCH STAGE STATUS: PASS
Unique completed configurations: 20
Family metrics used for model selection: NO


## Completion rule

Do not proceed to the next stage unless this notebook writes a valid `stage_manifest.json` with `PASS` or `PASS_WITH_NONBLOCKING_WARNINGS`. A hard integrity failure invalidates downstream execution until corrected and versioned.
